In [ ]:
import instagrapi 
import pandas as pd
import os
import collections
import asyncio
import copy
import time
import datetime
from pathlib import Path
import json, random, time
from dotenv import load_dotenv
load_dotenv()

import langchain
from langchain_openai import AzureChatOpenAI
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field
from operator import itemgetter

In [ ]:
openai_api_base_value = os.environ.get('openai_api_base_value')
openai_api_key_value = os.environ.get('openai_api_key_value')
openai_api_version_value = os.environ.get('openai_api_version_value')
deployment_name_value = os.environ.get('deployment_name_value')
openai_api_type_value = os.environ.get('openai_api_type_value')
ACCOUNT_USERNAME_VALUE = os.environ.get('ACCOUNT_USERNAME')
ACCOUNT_PASSWORD_VALUE = os.environ.get('ACCOUNT_PASSWORD')

In [ ]:
llm_model = AzureChatOpenAI(
                    model="gpt-4-1",
                    azure_endpoint=openai_api_base_value, 
                    openai_api_version=openai_api_version_value, 
                    deployment_name=deployment_name_value, 
                    openai_api_key=openai_api_key_value, 
                    timeout=120
                    )

In [ ]:
cl = instagrapi.Client()
cl.delay_range = [1, 7]

cl.set_device({
    "phone_manufacturer": "Google",
    "phone_model": "Pixel 7",
    "android_version": 34,
    "android_release": "14",
    "dpi": "560dpi",
    "resolution": "1080x2340"
})

if Path("session.json").exists():
    cl.load_settings("session.json")

try:
    cl.account_info()
except:
    cl.login(ACCOUNT_USERNAME_VALUE, ACCOUNT_PASSWORD_VALUE)
    cl.dump_settings("session.json")

In [ ]:
saved_post = cl.collections()

In [ ]:
collection_dict = {}
for _type in (saved_post):
    print(_type.name)
    if not _type.name in ['All Posts']:
        collection_name = _type.name
        collection_id = cl.collection_pk_by_name(collection_name)
        total_posts = _type.media_count
        print(f'collection name:{collection_name}, total posts: {total_posts}')
        collection_dict[collection_name] = cl.collection_medias(collection_pk=collection_id, amount = 0)

In [ ]:
collection_detail_dict = {}
collection_source_dict = {}
for key, value in collection_dict.items():
    collection_detail_lt = []
    collection_source_lt = []
    for detail in value:
        _content_id = detail.id
        _content = detail.caption_text
        _img = detail.image_versions2['candidates'][0]['url']
        _location = detail.location
        collection_detail_lt.append({'id':_content_id, 'content':_content})
        collection_source_lt.append({'id':_content_id, 'img':_img, 'location':_location,'raw_data':_content})
    collection_detail_dict[key] = (collection_detail_lt)
    collection_source_dict[key] = (collection_source_lt)

In [ ]:
class ll_answer(BaseModel):
    country: str = Field(description="country of the place")
    city: str = Field(description="city of the place")
    Zone: str = Field(description="Zone of the place")
    place_category: str = Field(description="what category of place")
    place_subcategory: str = Field(description="what subcategory of place")
    place_name: str = Field(description="name of the place")
    place_summary: str = Field(description="summary of the place")
    address_detail: str = Field(description="address of the place")
    opening_hour: str = Field(description="opening hour of the place")
    price_range: str = Field(description="price range of the place")

output_parser = JsonOutputParser(pydantic_object=ll_answer)

prompt = PromptTemplate(
    template="""
                You MUST ONLY return the result in below format, no additional is needed.
                {format_instructions}\n
                ==========================================================================
                {post_content}
                Base on the above information, please find the following data: 
                Country of the place (TRANSLATE the Country to Traditional Chinese), 
                City of the place (TRANSLATE the City to Traditional Chinese),
                Zone of the place (e.g., 銅鑼灣,秋葉原,旺角,心齋橋,原宿,尖沙咀,明洞)(TRANSLATE the Zone to Traditional Chinese),
                Category of the place (e.g., 餐廳,博物館,鄉村,公園,購物中心,活動,建築,飯店,景點,其他)(TRANSLATE the Category to Traditional Chinese),
                Subcategory of the place (e.g., 美術館,賞櫻,音樂節,壽喜燒,祭典,拉麵, *布丁 is special*)(TRANSLATE the Subcategory to Traditional Chinese),
                Name of the place, 
                Summary of the place in 50 words (TRANSLATE the Summary to Traditional Chinese),
                Address of the place,
                Opening hour of the place,
                Price range (Transfer the Price range to Hong Kong dollar based).\n
                -------------------------------------------
                Issue that you might faced:
                If you cannot find the above information, just fill in N/A.
                If there is multiple place in the information, please return multiple result separately.
                """,
    input_variables=["query"],
    partial_variables={"format_instructions": output_parser.get_format_instructions()},
)
chain = prompt | llm_model | output_parser

In [ ]:
async def async_invoke(content_detail):
    print('start invoke')
    result = {}
    result['id'] = content_detail['id']
    try:
        llm_result = await chain.ainvoke({"post_content": content_detail['content']})
        time.sleep(3)
        if type(llm_result)==list:
            _x = 1
            for multi_colls in llm_result:
                _id = content_detail['id'] + '_' + str(_x)
                multi_colls['id'] = content_detail['id']
                multi_colls['sub_id'] = _id
                _x+=1
            result = llm_result
        else:
            result.update(llm_result)
        print((result))
        return result
    except Exception as e:
        print(f'{content_detail['id']}->error:',  e)
        return None

async def main():
    result_dict = {}
    for key, value in collection_detail_dict.items():
        result_lt = []
        print(key)
        tasks = [async_invoke(content_detail) for content_detail in value]
        results = await asyncio.gather(*tasks)
        for result in results:
            if result is not None:
                result_lt.append(result)
        result_dict[key] = result_lt
    return result_dict

result_dict = await(main())

In [ ]:
temp_result_dict = copy.deepcopy(result_dict)
final_result_dict = {}
for key in temp_result_dict:
    final_result_dict[key] = []
    for i in range(len(temp_result_dict[key])):
        if type(temp_result_dict[key][i])==list:
            list_len = len(temp_result_dict[key][i])
            for j in range(list_len):
                temp_result_dict[key].append(temp_result_dict[key][i][j])
                final_result_dict[key].append(temp_result_dict[key][i][j])
            del temp_result_dict[key][i]
        else:
            final_result_dict[key].append(temp_result_dict[key][i])

In [ ]:
raw_df = pd.DataFrame([(Collection, value) for Collection, value in collection_source_dict.items()], columns=['Collection', 'value'])
raw_df = raw_df.explode('value')
raw_df = pd.concat([raw_df.drop('value', axis=1), raw_df['value'].apply(pd.Series)], axis=1)

In [ ]:
llm_df = pd.DataFrame([(Collection, value) for Collection, value in result_dict.items()], columns=['Collection', 'value'])
llm_df = llm_df.explode('value')
llm_df = pd.concat([llm_df.drop('value', axis=1), llm_df['value'].apply(pd.Series)], axis=1)

In [ ]:
final_df  = llm_df.merge(raw_df, left_on=['Collection','id'], right_on=['Collection','id'])
final_df['export_time'] = datetime.datetime.now()

In [ ]:
final_df

In [ ]:
final_df.to_excel("./travel_data.xlsx")